# 01 - F?rbered och utforska

Det h?r notebooket l?ser in verklig transaktionsdata, g?r en join mot en lookup-tabell och bygger en kundniv?tabell f?r clustering.

In [ ]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').exists() and (ROOT.parent / 'data').exists():
    ROOT = ROOT.parent
if not (ROOT / 'data').exists() and (ROOT.parent.parent / 'data').exists():
    ROOT = ROOT.parent.parent

transactions = pd.read_csv(ROOT / 'data' / 'raw' / 'transactions.csv', parse_dates=['InvoiceDate'])
regions = pd.read_csv(ROOT / 'data' / 'raw' / 'regions.csv')
transactions.head()

In [ ]:
transactions.shape, regions.shape

In [ ]:
quality = pd.DataFrame({
    'dtype': transactions.dtypes.astype(str),
    'missing_count': transactions.isna().sum(),
    'missing_pct': (transactions.isna().mean() * 100).round(2),
})
quality

In [ ]:
cleaned = transactions.dropna(subset=['CustomerID']).copy()
cleaned = cleaned[(cleaned['Quantity'] > 0) & (cleaned['UnitPrice'] > 0)].copy()
cleaned['CustomerID'] = cleaned['CustomerID'].astype(int)
cleaned['Country'] = cleaned['Country'].fillna('Unknown')
cleaned['Revenue'] = cleaned['Quantity'] * cleaned['UnitPrice']
cleaned['RegionGroup'] = cleaned['Country'].map(
    regions.set_index('Country')['RegionGroup'].to_dict()
).fillna('Other')
cleaned.head()

In [ ]:
invoice_level = (
    cleaned.groupby(['CustomerID', 'InvoiceNo'], as_index=False)
    .agg(invoice_total=('Revenue', 'sum'), invoice_date=('InvoiceDate', 'max'))
)
ref_date = cleaned['InvoiceDate'].max() + pd.Timedelta(days=1)

customers = (
    cleaned.groupby('CustomerID', as_index=False)
    .agg(
        last_purchase_date=('InvoiceDate', 'max'),
        frequency=('InvoiceNo', 'nunique'),
        monetary=('Revenue', 'sum'),
        basket_size=('Quantity', 'sum'),
        unique_products=('StockCode', 'nunique'),
        avg_unit_price=('UnitPrice', 'mean'),
        Country=('Country', 'first'),
        RegionGroup=('RegionGroup', 'first'),
    )
)
customers = customers.merge(
    invoice_level.groupby('CustomerID', as_index=False).agg(avg_order_value=('invoice_total', 'mean')),
    on='CustomerID',
    how='left',
)
customers['recency_days'] = (ref_date - customers['last_purchase_date']).dt.days
customers['avg_items_per_invoice'] = customers['basket_size'] / customers['frequency']
customers = customers[
    ['CustomerID', 'Country', 'RegionGroup', 'recency_days', 'frequency', 'monetary', 'avg_order_value', 'basket_size', 'avg_items_per_invoice', 'unique_products', 'avg_unit_price', 'last_purchase_date']
].sort_values('CustomerID').reset_index(drop=True)

customers.to_csv(ROOT / 'data' / 'raw' / 'customers.csv', index=False)
customers.to_csv(ROOT / 'data' / 'processed' / 'customer_enriched.csv', index=False)
customers.head()

In [ ]:
sns.set_theme(style='whitegrid', context='talk')
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, col in zip(axes.flat, ['recency_days', 'frequency', 'monetary', 'avg_order_value']):
    sns.histplot(data=customers, x=col, bins=24, element='step', stat='density', ax=ax, color='#1b9e77')
    ax.set_title(col.replace('_', ' ').title())
plt.tight_layout()
plt.show()

In [ ]:
corr_cols = ['recency_days', 'frequency', 'monetary', 'avg_order_value', 'basket_size', 'avg_items_per_invoice', 'unique_products', 'avg_unit_price']
plt.figure(figsize=(11, 8))
sns.heatmap(customers[corr_cols].corr(numeric_only=True), annot=True, fmt='.2f', cmap='Blues', center=0)
plt.title('Customer feature correlation heatmap')
plt.show()

Notebookens output ?r en kundniv?tabell i `data/processed/customer_enriched.csv`. Notebook 2 anv?nder samma schema f?r tr?ning och inference.